# Growing E. coli on Acetic Acid — Monod Kinetics in a Batch Bioreactor

**The situation:** [usecase 01](01_predict_ph_simple_liquid.ipynb) built the
KH₂PO₄ + NH₄Cl liquid and [usecase 02](02_equilibrate_with_atmospheric_gas.ipynb)
opened it to a sparged air atmosphere. Now inoculate that same vessel with
*E. coli*, using acetic acid as the sole carbon and energy source, and let it
grow. How fast does the population grow, how quickly is the acetate consumed,
does dissolved O₂ ever become limiting, and — since acetic acid is itself a
weak acid and the nitrogen source is ammonium — what happens to pH as growth
proceeds?

This is the first notebook in the series where the chemistry *evolves over
time* rather than being solved as a single equilibrium snapshot, so it
introduces the two pieces usecases 01-02 deliberately left out:
`ControlVolume` (a vessel of phases) and `Simulation` (the time-stepping
orchestrator). Everything else carries over unchanged — the same
`EquilibriumReaction` network from usecase 02, extended with exactly one new
building block: a `KineticReaction` for Monod growth, built via
`ReactionBuilder.monod_aerobic_growth()` and dropped into the same
`ReactionSystem` alongside the acid-base chemistry.

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import matplotlib.pyplot as plt

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemistry.species import Species
from PyOMES.chemistry.common_species import (
    H2O, H_plus, OH_minus,
    H3PO4, H2PO4_minus, HPO4_2minus, PO4_3minus,
    NH3, NH4_plus,
    CO2, HCO3_minus, CO3_2minus,
    K_plus, Cl_minus,
)
from PyOMES.chemistry import HenryEquilibrium
from PyOMES.reactions import (
    EquilibriumReaction, ReactionBuilder, ReactionSystem, StoichiometryEntry,
)
from PyOMES.core import (
    ControlVolume, GasPhase, LiquidPhase,
    KineticTransferModel, EquilibriumTransferModel,
    Simulation, GasFeed, PressureReliefVent,
)
from PyOMES.core.phases import R_L_ATM_MOL_K

# The explicit-Euler CV solver clamps a species' removal rate when a step
# would otherwise drive it negative (AccuracyWarning), and separately flags
# any resulting element/charge drift above its tolerance (ConservationWarning).
# Both are the solver's self-correcting safety net doing its job at dt_h=0.01,
# not a sign of a wrong answer here (Section 6 checks that directly) --
# silenced only so the run cell's output stays readable.
from PyOMES.monitoring.accuracy import AccuracyWarning
from PyOMES.monitoring.conservation import ConservationWarning
warnings.filterwarnings("ignore", category=AccuracyWarning)
warnings.filterwarnings("ignore", category=ConservationWarning)

def _e(sp, coeff, phase="liquid"):
    return StoichiometryEntry(species=sp, phase=phase, coefficient=coeff)

print("Imports OK")


## 1  Literature-grounded Monod parameters

*E. coli* grows on acetate far more slowly than on glucose — acetate uptake
feeds the glyoxylate shunt and gluconeogenesis rather than glycolysis, and
that costs growth rate. For reference, *E. coli*'s Monod constant on
**glucose** is well characterised: μ_max ≈ 0.66 h⁻¹, K_s ≈ 2-4 mg/L
([BioNumbers ID 111049](https://bionumbers.hms.harvard.edu/bionumber.aspx?id=111049)).
Acetate kinetics are reported far less consistently — Kovarova-Kovar & Egli
(1998, *Microbiol. Mol. Biol. Rev.* 62:646-666) is the standard review
explaining *why*: reported K_s values for a given substrate vary by orders
of magnitude across studies because K_s depends on the cells' physiological
state, not just the transporter's intrinsic affinity. The parameters below
are chosen to be *representative* of the reported ranges rather than a
single paper's point estimate — treat them as a reasonable starting model,
not a re-derivation of any one study.

| Parameter | Value | Basis |
|---|---|---|
| μ_max | 0.30 h⁻¹ | Representative of aerobic minimal-medium growth on acetate as sole carbon source — well below glucose's 0.66 h⁻¹, consistent with the glyoxylate-shunt/gluconeogenesis cost noted above. |
| K_s | 5 mg/L acetate | Same order of magnitude as *E. coli*'s high-affinity glucose K_s (2-4 mg/L); high-affinity organic-acid transporters place acetate K_s in a similar low-mg/L range. |
| Y_X/S | 0.36 g DCW/g acetate | Within the range reported for aerobic *E. coli* on acetate — Leone et al. (2015, *Microb. Cell Fact.* 14:106) measured Y_X/S = 0.39 ± 0.01 g/g for *E. coli* BL21 under controlled pH/DO; other studies report lower yields (0.17-0.30 g/g) at low growth rates, consistent with a Pirt-type maintenance cost. |
| K_O2 | 0.2 mg/L | Typical high-affinity dissolved-O2 half-saturation constant for aerobic heterotrophs (same value used in this repository's own `ReactionBuilder.monod_aerobic_growth` docstring example). |

Biomass itself is represented by the generic empirical formula
**CH₁.₈O₀.₅N₀.₂** (MW ≈ 24.6 g per C-mol) — the average bacterial biomass
composition widely used in bioprocess engineering (Roels, 1983,
*Energetics and Kinetics in Biotechnology*) when a strain-specific formula
isn't available. It plays the same role here that the generic yeast formula
(CH₁.₆₁O₀.₅₆N₀.₁₆) played in `demos/builder/batch_fermenter.py`.

In [ ]:
mu_max_per_h = 0.30    # h^-1
Ks_gL        = 5e-3    # g acetate / L
Yxs          = 0.36    # g biomass / g acetate
Ko2_gL       = 0.2e-3  # g O2 / L

print(f"mu_max = {mu_max_per_h} /h, Ks = {Ks_gL*1e3} mg/L, "
      f"Yxs = {Yxs} g/g, Ko2 = {Ko2_gL*1e3} mg/L")


## 2  Declare the chemistry

Same seven reactions as usecase 02 (water, the phosphate ladder, ammonium,
the carbonate ladder), plus one new one: acetic acid's own dissociation.
It's a weak acid exactly like phosphoric acid was in usecase 01 — just a
single step instead of three, with a pK_a (4.756) that happens to sit close
to phosphate's amphoteric pH (½(pK_a1+pK_a2) ≈ 4.68 from usecase 01 §4),
so the two acids interact rather than one simply dominating.

| Reaction | log K | p$K_a$ | Role |
|---|---|---|---|
| H₂O ⇌ H⁺ + OH⁻ | −14.0 | 14.0 | water autoionization |
| H₃PO₄ ⇌ H₂PO₄⁻ + H⁺ | −2.15 | 2.15 | phosphate, 1st step |
| H₂PO₄⁻ ⇌ HPO₄²⁻ + H⁺ | −7.20 | 7.20 | phosphate, 2nd step |
| HPO₄²⁻ ⇌ PO₄³⁻ + H⁺ | −12.35 | 12.35 | phosphate, 3rd step |
| NH₄⁺ ⇌ NH₃ + H⁺ | −9.25 | 9.25 | ammonium/ammonia |
| CO₂(aq) + H₂O ⇌ HCO₃⁻ + H⁺ | −6.35 | 6.35 | carbonate, 1st step |
| HCO₃⁻ ⇌ CO₃²⁻ + H⁺ | −10.33 | 10.33 | carbonate, 2nd step |
| AceticAcid ⇌ Acetate⁻ + H⁺ | −4.756 | 4.756 | the substrate's own acid-base chemistry |

`AceticAcid` and `Acetate-` aren't in `common_species` (a substrate-specific
pair, not a universal inorganic), so they're declared locally — the same
pattern `demos/model_api/chemistry/reaction_system.py` uses. `Ecoli` is
declared the same way, carrying the CH₁.₈O₀.₅N₀.₂ formula from §1.

In [ ]:
ACETIC_ACID = Species(
    id="AceticAcid", atoms={"C": 2, "H": 4, "O": 2}, charge=0, MW=60.052,
)
ACETATE_MINUS = Species(
    id="Acetate-", atoms={"C": 2, "H": 3, "O": 2}, charge=-1, MW=59.044,
)
ECOLI = Species(
    id="Ecoli", atoms={"C": 1, "H": 1.8, "O": 0.5, "N": 0.2}, charge=0,
    MW=24.626,
)

water = EquilibriumReaction(
    stoichiometry=[_e(H2O, -1), _e(H_plus, +1), _e(OH_minus, +1)],
    log_K=-14.0, label="water",
)
p1 = EquilibriumReaction(
    stoichiometry=[_e(H3PO4, -1), _e(H2PO4_minus, +1), _e(H_plus, +1)],
    log_K=-2.15, total_id="H3PO4", label="p1",
)
p2 = EquilibriumReaction(
    stoichiometry=[_e(H2PO4_minus, -1), _e(HPO4_2minus, +1), _e(H_plus, +1)],
    log_K=-7.20, total_id="H3PO4", label="p2",
)
p3 = EquilibriumReaction(
    stoichiometry=[_e(HPO4_2minus, -1), _e(PO4_3minus, +1), _e(H_plus, +1)],
    log_K=-12.35, total_id="H3PO4", label="p3",
)
nh4 = EquilibriumReaction(
    stoichiometry=[_e(NH4_plus, -1), _e(NH3, +1), _e(H_plus, +1)],
    log_K=-9.25, total_id="NH3", label="nh4",
)
co2_first = EquilibriumReaction(
    stoichiometry=[_e(CO2, -1), _e(H2O, -1), _e(HCO3_minus, +1), _e(H_plus, +1)],
    log_K=-6.35, total_id="CO2", label="co2_first",
)
co2_second = EquilibriumReaction(
    stoichiometry=[_e(HCO3_minus, -1), _e(CO3_2minus, +1), _e(H_plus, +1)],
    log_K=-10.33, total_id="CO2", label="co2_second",
)
acetate_eq = EquilibriumReaction(
    stoichiometry=[_e(ACETIC_ACID, -1), _e(ACETATE_MINUS, +1), _e(H_plus, +1)],
    log_K=-4.756, total_id="AceticAcid", label="acetate_eq",
)

print("8 equilibrium reactions declared.")


## 3  Declare the Monod growth kinetics

`ReactionBuilder.monod_aerobic_growth()` is the highest-level tool for this
job — the same role `NRChemicalEquilibriumEngine.from_reactions()` played
for the acid-base network in usecase 01. Given the substrate and biomass
`Species`, the four kinetic parameters from §1, and `balance="CHNO"`, it
derives the O₂, NH₃ (nitrogen source), CO₂, and H₂O coefficients from
elemental balance — the same closure `ReactionBuilder.aerobic_growth` uses
internally, so the growth reaction is automatically self-consistent on C,
H, N, and O. `n_source_id` defaults to `"NH3"`, so growth draws directly on
the *same* ammoniacal-nitrogen pool the `nh4` reaction above manages —
biomass synthesis and the ammonium/ammonia equilibrium share one total.

In [ ]:
growth = ReactionBuilder.monod_aerobic_growth(
    substrate=ACETIC_ACID, biomass=ECOLI,
    mu_max_per_h=mu_max_per_h, Ks_gL=Ks_gL, yield_gX_gS=Yxs, Ko2_gL=Ko2_gL,
    balance="CHNO", label="growth_on_AceticAcid",
)

print("Derived stoichiometry (per mol AceticAcid consumed):")
for entry in growth.stoichiometry:
    print(f"  {entry.coefficient:+.4f}  {entry.species.id:<12} ({entry.phase})")


## 4  Assemble the batch vessel and run

All nine reactions (eight equilibria + the one kinetic growth reaction) go
into a single `ReactionSystem`. The vessel is a 2 L bottle at 37 °C
(typical *E. coli* culture temperature), continuously sparged with air —
`GasFeed` supplies fresh O₂/N₂/CO₂ the same way usecase 02's atmosphere did,
but `GasFeed` only ever adds gas, so a `PressureReliefVent` is needed
alongside it to vent the excess and hold the headspace at 1 atm (mirroring
`demos/builder/cstr_fermenter.py`'s vessel plumbing). O₂ and CO₂ transfer
kinetically (kLa = 150 /h, the same value used throughout
`demos/builder/`); N₂ stays at instantaneous Henry equilibrium since
nothing reacts with it.

The liquid starts at usecase 01's M9-like recipe (22 mM KH₂PO₄, 18.7 mM
NH₄Cl) plus 4 g/L acetic acid (0.4% w/v — the same loading Leone et al.
used) and a small inoculum (0.05 g/L biomass). `K+`/`Cl-` are the actual
counter-ions each salt contributes — the charge-balance solver needs them
declared as real species in `n_mol`, the CV-driven equivalent of usecase
01's `strong_ions=` argument on a bare `engine.solve()` call.

In [ ]:
system = ReactionSystem(
    [water, p1, p2, p3, nh4, co2_first, co2_second, acetate_eq, growth],
    label="ecoli_on_acetate",
)

V_total_L = 2.0
headspace_frac = 0.20
V_liq = V_total_L * (1.0 - headspace_frac)
V_gas = V_total_L * headspace_frac
T_K = 310.15   # 37 C

n_total_gas = (1.0 * V_gas) / (R_L_ATM_MOL_K * T_K)   # 1 atm headspace
gas_phase = GasPhase(
    n_mol={
        "O2": n_total_gas * 0.2095,
        "N2": n_total_gas * 0.7901,
        "CO2": n_total_gas * 0.0004,
    },
    V_L=V_gas, T_K=T_K,
)

CT_P = 0.022    # mol/L KH2PO4 (usecase 01's recipe)
CT_N = 0.0187   # mol/L NH4Cl
C_AcOH0_gL = 4.0                        # g/L acetic acid (0.4% w/v)
C_AcOH0 = C_AcOH0_gL / ACETIC_ACID.MW
X0_gL = 0.05                             # g/L inoculum
X0 = X0_gL / ECOLI.MW

liquid_phase = LiquidPhase(
    n_mol={
        "H3PO4": CT_P * V_liq, "NH3": CT_N * V_liq,
        "AceticAcid": C_AcOH0 * V_liq, "Ecoli": X0 * V_liq,
        "K+": CT_P * V_liq, "Cl-": CT_N * V_liq,
        "O2": 0.0, "CO2": 0.0, "N2": 0.0,
    },
    V_L=V_liq, T_K=T_K,
)

transfer_models = {
    "O2": KineticTransferModel(
        partition_model=HenryEquilibrium(H_ref=1.3e-5, dlnH=1500.0), k_transfer=150.0,
    ),
    "CO2": KineticTransferModel(
        partition_model=HenryEquilibrium(H_ref=3.4e-4, dlnH=2400.0), k_transfer=150.0 * 0.9,
    ),
    "N2": EquilibriumTransferModel(HenryEquilibrium(H_ref=6.4e-6, dlnH=1300.0)),
}
gas_feed = GasFeed(
    vvm_min=1.0, y={"O2": 0.2095, "N2": 0.7901, "CO2": 0.0004}, P_inlet_atm=1.0,
    phase_key="gas", liquid_phase_key="liquid", label="gas_feed",
)
vent = PressureReliefVent(P_set_atm=1.0, mode="instant")

cv = ControlVolume(
    phases={"gas": gas_phase, "liquid": liquid_phase},
    transfer_models=transfer_models,
    boundaries=[gas_feed, vent],
    reaction_system=system,
    label="batch_ecoli",
)

sim = Simulation(cvs={"main": cv}, label="usecase03")
tau_h, n_steps = 20.0, 2000
result = sim.run(tau_h=tau_h, n_steps=n_steps)

print(f"Ran {tau_h:.0f} h ({n_steps} steps) in {result.runtime_s:.2f} s wall-clock.")


## 5  Biomass, substrate, dissolved O₂, and pH over the batch

Four views of the same 20-hour run: acetate depletes as biomass grows,
dissolved O₂ stays close to air saturation throughout (§7 checks whether
it ever actually limits growth), and pH swings noticeably — starting well
below usecase 01's phosphate-only baseline (≈4.73) because 4 g/L acetic
acid is itself a substantial acid load, comparable in magnitude to the
22 mM phosphate buffer meant to resist it, then recovering toward that
baseline as the acid is consumed.

In [ ]:
cv_key = "main"
t = result.t_h
X = result.liquid_mol[cv_key]["Ecoli"] * ECOLI.MW / V_liq
S = result.liquid_mol[cv_key]["AceticAcid"] * ACETIC_ACID.MW / V_liq
DO_mgL = result.liquid_mol[cv_key]["O2"] / V_liq * 32e3
pH = result.pH[cv_key]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].plot(t, X, color="tab:green")
axes[0, 0].set_xlabel("time (h)")
axes[0, 0].set_ylabel("biomass (g/L)")
axes[0, 0].set_title("E. coli growth")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(t, S, color="tab:blue")
axes[0, 1].set_xlabel("time (h)")
axes[0, 1].set_ylabel("acetic acid (g/L)")
axes[0, 1].set_title("Substrate depletion")
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(t, DO_mgL, color="tab:red")
axes[1, 0].set_xlabel("time (h)")
axes[1, 0].set_ylabel("dissolved O2 (mg/L)")
axes[1, 0].set_title("Dissolved oxygen")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(t, pH, color="tab:purple")
axes[1, 1].axhline(4.734, color="gray", ls="--", lw=1,
                    label="usecase 01 baseline (no acetate), pH=4.73")
axes[1, 1].set_xlabel("time (h)")
axes[1, 1].set_ylabel("pH")
axes[1, 1].set_title("pH")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Biomass:  {X[0]:.3f} -> {X[-1]:.3f} g/L")
print(f"Acetate:  {S[0]:.3f} -> {S[-1]:.3f} g/L")
print(f"DO:       {DO_mgL[0]:.3f} -> {DO_mgL[-1]:.3f} mg/L "
      f"(min {DO_mgL.min():.3f} mg/L)")
print(f"pH:       {pH[0]:.3f} -> {pH[-1]:.3f} (min {np.nanmin(pH):.3f})")


## 6  Validating the run against the declared model

Two independent checks that the simulator is actually solving the model
that was declared in §§3-4, not just producing a plausible-looking curve:

- **Yield mass balance.** `ReactionBuilder.aerobic_growth` fixed the ratio
  of biomass produced to substrate consumed at exactly `Yxs` when it built
  the stoichiometry — every mol of `AceticAcid` consumed produces `Y_mol`
  mol of `Ecoli`, no more, no less. So `Yxs * (S₀ - S_f)` should equal
  `X_f - X₀` to numerical precision, independent of `mu_max`, `Ks`, or
  `Ko2` (those only control the *rate*, not the *stoichiometry*).
- **Monod self-consistency.** The rate function inside `growth` computes
  `μ = mu_max · S/(Ks+S) · O2/(Ko2+O2)` from whatever `S` and `O2` the
  solver hands it at each step. Recomputing that same formula from the
  *recorded* `S`/`DO` trajectory and comparing it to `μ` backed out
  numerically from `dX/dt / X` checks that the integrator is actually
  tracking the declared rate law, not some subtly different one.

In [ ]:
# Yield mass balance
dS = S[0] - S[-1]
dX = X[-1] - X[0]
print(f"Yxs * (S0 - Sf) = {Yxs * dS:.4f} g/L")
print(f"Xf - X0         = {dX:.4f} g/L")
print(f"Agreement to {abs(Yxs*dS - dX):.2e} g/L\n")

# Monod self-consistency: mu backed out of the recorded trajectory
# vs. mu recomputed from the declared rate law at that same trajectory.
mu_numeric = np.gradient(X, t) / np.maximum(X, 1e-9)
DO_gL = DO_mgL / 1e3
mu_monod = mu_max_per_h * (S / (Ks_gL + S)) * (DO_gL / (Ko2_gL + DO_gL))

# Restrict the comparison to the bulk growth phase, away from both its
# edges: near t=0, dissolved O2 starts at exactly 0 mol/L (not yet
# gas-liquid-equilibrated), so the central-difference dX/dt right at that
# one-step transient understates the true instantaneous rate; and near
# substrate exhaustion, S/(Ks+S) has its steepest curvature (Ks is tiny
# next to S), where any finite-difference derivative is least accurate.
# Both are numerical-differentiation edge effects, not signs the
# integrator is tracking the wrong rate law -- the mid-growth agreement
# below is the actual check.
growing = (S > 0.05) & (t > 0.02)
max_gap = np.max(np.abs(mu_numeric[growing] - mu_monod[growing]))
print(f"Max |mu_numeric - mu_monod| over the growth phase: {max_gap:.5f} /h")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t[growing], mu_numeric[growing], "o", ms=3, color="tab:orange", label="mu, from dX/dt / X")
ax.plot(t[growing], mu_monod[growing], "-", color="tab:blue", label="mu, from Monod formula")
ax.set_xlabel("time (h)")
ax.set_ylabel("specific growth rate (1/h)")
ax.set_title("Monod self-consistency check")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Dissolved O₂ stayed within a few percent of air saturation for the whole
run (§5's DO panel) — at `kLa(O2)=150 /h`, oxygen transfer keeps up with
this culture's peak O₂ demand comfortably, so the `Ko2_gL` Monod term
never actually binds here (`DO >> Ko2` throughout). That's a real result,
not an oversight: it takes either a much lower kLa or a much denser
culture before O₂ transfer becomes the growth-limiting step, which is
exactly the tradeoff `demos/builder/cstr_fermenter.py`'s `DOAgitationController`
exists to manage.

## 7  Sensitivity: μ_max sets the timescale, not the final yield

The mass-balance check in §6 already hinted at this: `mu_max` never
entered the yield relationship. Rerunning the same batch at three
different `mu_max` values — holding `Ks`, `Yxs`, `Ko2`, and every initial
condition fixed — should show growth curves that reach the *same* final
biomass and substrate concentrations, just on different timescales. This
is the batch-culture analogue of usecase 02 §4's sensitivity sections: one
parameter swept, one qualitative expectation from the model's structure
checked directly against the simulation.

In [ ]:
def run_batch(mu_max_test):
    growth_i = ReactionBuilder.monod_aerobic_growth(
        substrate=ACETIC_ACID, biomass=ECOLI,
        mu_max_per_h=mu_max_test, Ks_gL=Ks_gL, yield_gX_gS=Yxs, Ko2_gL=Ko2_gL,
        balance="CHNO", label="growth_on_AceticAcid",
    )
    system_i = ReactionSystem(
        [water, p1, p2, p3, nh4, co2_first, co2_second, acetate_eq, growth_i],
        label="ecoli_on_acetate",
    )
    gas_phase_i = GasPhase(
        n_mol={
            "O2": n_total_gas * 0.2095, "N2": n_total_gas * 0.7901,
            "CO2": n_total_gas * 0.0004,
        },
        V_L=V_gas, T_K=T_K,
    )
    liquid_phase_i = LiquidPhase(
        n_mol={
            "H3PO4": CT_P * V_liq, "NH3": CT_N * V_liq,
            "AceticAcid": C_AcOH0 * V_liq, "Ecoli": X0 * V_liq,
            "K+": CT_P * V_liq, "Cl-": CT_N * V_liq,
            "O2": 0.0, "CO2": 0.0, "N2": 0.0,
        },
        V_L=V_liq, T_K=T_K,
    )
    cv_i = ControlVolume(
        phases={"gas": gas_phase_i, "liquid": liquid_phase_i},
        transfer_models=transfer_models,
        boundaries=[
            GasFeed(vvm_min=1.0, y={"O2": 0.2095, "N2": 0.7901, "CO2": 0.0004},
                    P_inlet_atm=1.0, phase_key="gas", liquid_phase_key="liquid"),
            PressureReliefVent(P_set_atm=1.0, mode="instant"),
        ],
        reaction_system=system_i, label=f"mu{mu_max_test}",
    )
    sim_i = Simulation(cvs={"main": cv_i}, label=f"sweep_mu{mu_max_test}")
    return sim_i.run(tau_h=tau_h, n_steps=n_steps)

mu_max_values = [0.15, 0.30, 0.60]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for mu_test in mu_max_values:
    res_i = run_batch(mu_test)
    X_i = res_i.liquid_mol["main"]["Ecoli"] * ECOLI.MW / V_liq
    S_i = res_i.liquid_mol["main"]["AceticAcid"] * ACETIC_ACID.MW / V_liq
    axes[0].plot(res_i.t_h, X_i, label=f"mu_max = {mu_test}/h")
    axes[1].plot(res_i.t_h, S_i, label=f"mu_max = {mu_test}/h")

axes[0].set_xlabel("time (h)")
axes[0].set_ylabel("biomass (g/L)")
axes[0].set_title("Biomass: same asymptote, different timescale")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel("time (h)")
axes[1].set_ylabel("acetic acid (g/L)")
axes[1].set_title("Substrate: same depletion, different timescale")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Where to go next

- **Closed-loop pH and DO control** — this notebook lets pH and DO drift
  freely; `demos/builder/batch_fermenter.py` (`PHController`) and
  `demos/builder/cstr_fermenter.py` (`PHController` + `DOAgitationController`
  cascade) show how `PyOMES.control.cv_loops` closes the loop on both.
- **Continuous culture (chemostat) instead of batch** — the same organism
  and substrate at steady state under continuous feed/drain, including the
  analytical Monod steady-state check — see
  `demos/builder/cstr_fermenter.py`.
- **The declarative building blocks used here in isolation** —
  `demos/model_api/chemistry/reaction_system.py` walks through
  `KineticReaction`, `EquilibriumReaction`, and `ReactionSystem` one at a
  time without running a `Simulation`.
- **`FermenterBuilder`, the fluent alternative to assembling `ControlVolume`
  by hand** — `models/vlmodels/fermenter/config/builder.py` wraps vessel
  geometry, gas feed, transfer, organism, and substrate into one chained
  call; §4 of this notebook is what it builds internally.